# 用 Sciverse 下载论文图表资源

> 从全文 Markdown 中提取图表路径，通过 resource 接口获取二进制文件

**场景**: 用户需要提取论文中的图表（如实验结果图、流程图、表格截图）用于报告、演示或多模态 RAG。

**使用接口**: content, resource

**预估调用量**: ~3–10 次 API 调用

---


## Step 1: 环境准备

安装依赖并配置环境变量


In [ ]:
!pip install httpx
import os
os.environ["SCIVERSE_API_TOKEN"] = "sv-your-token-here"  # 替换为你的真实值


## Step 2: 从全文中提取图表路径

content 返回的 Markdown 中，图表以标准 Markdown 图片语法引用


In [ ]:
import os
import re
import asyncio
import httpx
from pathlib import Path

BASE = "https://api.sciverse.space"
TOKEN = os.environ["SCIVERSE_API_TOKEN"]
HEADERS = {"Authorization": f"Bearer {TOKEN}"}

async def get_content(doc_id: str, offset: int = 0, limit: int = 4000):
    async with httpx.AsyncClient(timeout=30) as client:
        resp = await client.get(
            f"{BASE}/content", headers=HEADERS,
            params={"doc_id": doc_id, "offset": offset, "limit": limit}
        )
        resp.raise_for_status()
        return resp.json()

async def main():
    result = await get_content("af2_nature_2021", offset=0, limit=4000)
    # 注意：响应字段是 text
    markdown_text = result["text"]
    # 提取所有图片路径
    figure_paths = re.findall(r'!\\[.*?\\]\\((.*?)\\)', markdown_text)
    print(f"Found {len(figure_paths)} figures:")
    for p in figure_paths:
        print(f"  {p}")
    return figure_paths

figure_paths = asyncio.run(main())


## Step 3: 调用 resource 下载图表

对每个路径调用 resource 接口获取二进制数据。参数是 file_name（非 path）


In [ ]:
async def download_resource(file_name: str, save_dir: str = "./figures"):
    """下载资源文件。参数 file_name 为 content 中提取的相对路径"""
    Path(save_dir).mkdir(exist_ok=True)
    async with httpx.AsyncClient(timeout=60) as client:
        resp = await client.get(
            f"{BASE}/resource",
            headers=HEADERS,
            params={"file_name": file_name}  # 注意：参数是 file_name
        )
        resp.raise_for_status()
        local_name = file_name.split("/")[-1]
        save_path = f"{save_dir}/{local_name}"
        Path(save_path).write_bytes(resp.content)
        print(f"  Saved: {save_path} ({len(resp.content)} bytes)")
        return save_path

async def download_all(paths: list):
    results = []
    for p in paths:
        try:
            saved = await download_resource(p)
            results.append(saved)
        except httpx.HTTPStatusError as e:
            print(f"  Failed: {p} ({e.response.status_code})")
    return results

saved_files = asyncio.run(download_all(figure_paths))


## 注意事项

- resource 接口参数是 file_name（不是 path），传入 content 中提取的相对路径即可
- resource 接口返回原始二进制流，Content-Type 为实际 MIME 类型
- 图表路径格式通常为 dt=文献ID/p_页码/文件名，由 content 接口给出
- 部分文档可能没有图表资源（resource 返回 404），需做异常处理
- 建议在 Agent 侧缓存已下载的图表，避免重复请求


---

## 下一步

- [申请 API Token](https://sciverse.opendatalab.com/docs#auth)
- [查看完整 API 文档](https://sciverse.opendatalab.com/docs#sciverse/api)
- [更多 Cookbook 案例](https://sciverse.opendatalab.com/docs#cookbook)
